# Querying ERA5 with SQL

Edit the query, check its plan, then run it on the deployed lambdas.

In [ ]:
import os

from zarr_datafusion_lambda import sql

# Where ERA5 and the workers are. Read once a kernel, at the first query.
os.environ["AWS_REGION"] = "us-east-1"

tables = {"era5": {"url": "s3://earthmover-icechunk-era5/icechunkV2", "group": "single/spatial"}}

In [ ]:
query = """
SELECT date_trunc('day', CAST(valid_time AS TIMESTAMP))                 AS day,
     sum(t2m * cos(radians(latitude))) / sum(cos(radians(latitude))) AS mean_t2m
FROM era5
WHERE valid_time >= TIMESTAMP '2020-01-01' AND valid_time < TIMESTAMP '2020-01-08'
GROUP BY 1
ORDER BY 1
"""

In [ ]:
print(sql.explain(query, tables))

In [ ]:
result = sql.run(query, tables, progress=True)
df = result.table.to_pandas()
print(f"{result.chunks} chunks in {result.invocations} invocations")
df

## Running in the client

A query fans out whenever the planner in zarr-datafusion-lambda can do so. The
`lambda.enabled` option set to `false` runs the whole query in zarr-datafusion, as SQL's
`SET lambda.enabled = false` would; `true`, the default, fans out on lambdas.

This compares the two on a point query over the time-series layout, which reads only a chunk or two.

In [ ]:
point_tables = {
    "era5": {"url": "s3://earthmover-icechunk-era5/icechunkV2", "group": "single/temporal"}
}
point_query = """
SELECT date_trunc('month', CAST(valid_time AS TIMESTAMP)) AS month,
       avg(t2m)                                          AS mean_t2m
FROM era5
WHERE latitude = 51.5 AND longitude = 0
  AND valid_time >= TIMESTAMP '2020-01-01' AND valid_time < TIMESTAMP '2021-01-01'
GROUP BY 1
ORDER BY 1
"""
in_client = {"lambda.enabled": "false"}

In [ ]:
print(sql.explain(point_query, point_tables))

In [ ]:
print(sql.explain(point_query, point_tables, config=in_client))

In [ ]:
%%time
result = sql.run(point_query, point_tables, progress=True)
df = result.table.to_pandas()
print(f"{result.chunks} chunks in {result.invocations} invocations")
df

In [ ]:
%%time
sql.run(point_query, point_tables, config=in_client).table.to_pandas()